# Benchmark détection + modélisation des chirps

Ce notebook compare `bat_chirp_annotations.json` avec le détecteur SNR/blob et le pipeline de modélisation courant.

**Version actuelle :** le gate absolu historique `max_value < 0.7` est supprimé dans `bat_analysis/modelling.py`. Tous les autres seuils de modélisation restent inchangés.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from tkinter import Tk, filedialog
import pandas as pd

from benchmark import run_benchmark
from benchmark.diagnostics import print_modelling_diagnostics


## Localiser automatiquement le projet

In [ ]:
def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'bat_analysis').exists():
            return candidate
    raise FileNotFoundError('Impossible de trouver la racine du projet bat_full_spectrum_processing')

PROJECT_ROOT = find_project_root()
analysis_py = PROJECT_ROOT / 'bat_analysis' / 'modelling.py'

print('Projet      :', PROJECT_ROOT)
print('Traitement  :', analysis_py)
print('Existe      :', analysis_py.exists())


## Choisir le JSON d'annotation

In [ ]:
root = Tk()
root.withdraw()
root.attributes('-topmost', True)
annotation_json = filedialog.askopenfilename(
    title='Sélectionner bat_chirp_annotations.json',
    filetypes=[('JSON', '*.json'), ('Tous les fichiers', '*.*')],
)
root.destroy()

annotation_json = Path(annotation_json)
print('Annotations :', annotation_json)


## Lancer le benchmark complet

Le matching annotation ↔ détection utilise les mêmes paramètres que lors de la baseline précédente.

In [ ]:
result = run_benchmark(
    annotation_json=annotation_json,
    analysis_py=analysis_py,
    min_iou=0.05,
    max_center_error_ms=4.0,
    verbose=True,
)


## Résumé global

In [ ]:
pd.Series(result.summary)


## Résultats par WAV

In [ ]:
result.files

## Résultats par chirp

`failure_stage` distingue les erreurs de détection des éventuels échecs de modélisation.

In [ ]:
cols = [
    'relative_path', 'chirp_id', 'detected', 'model_success', 'failure_stage',
    'manual_start_ms', 'manual_end_ms', 'manual_duration_ms',
    'candidate_time_mid_ms', 'detection_center_error_ms', 'detection_iou',
    'median_abs_error_khz', 'mae_khz', 'rmse_khz', 'p95_abs_error_khz', 'coverage',
    'start_error_ms', 'end_error_ms'
]
result.chirps[[c for c in cols if c in result.chirps.columns]]

## Chirps encore en échec

In [ ]:
failures = result.chirps[result.chirps['failure_stage'].fillna('') != ''].copy()
print(f'{len(failures)} chirp(s) en échec sur {len(result.chirps)} chirps manuels')
failures

## Diagnostic succès / échecs de modélisation

Cette cellule reste utile si des TP détectés échouent encore après suppression du gate 0.7.

In [ ]:
diag = print_modelling_diagnostics(result)


## Comparaison avec la baseline historique

Baseline avant suppression du gate 0.7 : 158 chirps manuels, 113 TP, 28 FP, 45 FN, 50 modélisations réussies, recall end-to-end 31.65 %.

In [ ]:
baseline = {
    'true_positives': 113,
    'false_positives': 28,
    'false_negatives': 45,
    'precision': 0.801418,
    'recall': 0.715190,
    'f1': 0.755853,
    'model_success': 50,
    'model_success_rate_on_tp': 0.442478,
    'end_to_end_recall': 0.316456,
}

compare_keys = list(baseline.keys())
comparison = pd.DataFrame({
    'baseline_gate_0.7': pd.Series(baseline),
    'current_no_gate': pd.Series({k: result.summary.get(k) for k in compare_keys}),
})
comparison['delta'] = comparison['current_no_gate'] - comparison['baseline_gate_0.7']
comparison


## Sauvegarder les résultats

In [ ]:
output_dir = Path(annotation_json).parent / 'benchmark_results' / 'no_amplitude_gate'
result.save_csv(output_dir)
comparison.to_csv(output_dir / 'comparison_vs_gate_0_7.csv')
print('Résultats sauvegardés dans :', output_dir)
